<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.3-embeddings-and-vector-stores/notebooks/exercises-6.3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.3 — Embeddings & Vector Stores
Netsetos GenAI Course v2.0 | Module 6

MTEB rankings, Matryoshka, vector DB ecosystem, indexing algorithms, production patterns


In [ ]:
# pip install chromadb sentence-transformers qdrant-client openai -q
print('Ready for embeddings & vector stores')


## Ex 1: Embedding Model Comparison


In [ ]:
from openai import OpenAI
import numpy as np

client = OpenAI()

def embed(texts, model='text-embedding-3-small'):
    resp = client.embeddings.create(input=texts, model=model)
    return [e.embedding for e in resp.data]

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

texts = ['machine learning algorithms', 'AI deep learning', 'biryani recipe']
vecs = embed(texts)

for i in range(len(texts)):
    for j in range(i+1, len(texts)):
        sim = cosine_sim(vecs[i], vecs[j])
        print(f'  {texts[i][:25]:25s} ↔ {texts[j][:25]:25s} = {sim:.3f}')

print(f'\nDimensions: {len(vecs[0])}')
print(f'Model: text-embedding-3-small | Cost: $0.02/M tokens')


## Ex 2: Matryoshka Dimensions


In [ ]:
# OpenAI supports Matryoshka via dimensions parameter
client = OpenAI()

query = 'What is retrieval augmented generation?'
dims = [256, 512, 1024, 1536]

for d in dims:
    resp = client.embeddings.create(
        input=[query], model='text-embedding-3-small', dimensions=d)
    vec = resp.data[0].embedding
    print(f'  {d:5d} dims: norm={np.linalg.norm(vec):.3f}, first 5={[round(v,4) for v in vec[:5]]}')

print('\nKey insight: 256 dims from 3-large outperforms ada-002 at 1536!')
print('Two-stage: shortlist with 256 (fast), rerank with full (precise)')
print('Storage: 768 dims = 75% savings vs 3072')


## Ex 3: ChromaDB Quick Start


In [ ]:
import chromadb

client = chromadb.Client()
collection = client.create_collection('demo', metadata={'hnsw:space': 'cosine'})

# Add documents
collection.add(
    documents=['RAG grounds LLM responses in external documents.',
               'Chunking at 512 tokens gives best accuracy.',
               'Hybrid search improves NDCG by 26-31%.'],
    ids=['doc1', 'doc2', 'doc3']
)

# Query
results = collection.query(query_texts=['What chunk size is best?'], n_results=2)
for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f'  [{dist:.3f}] {doc[:60]}...')

print(f'\nChromaDB: {collection.count()} docs indexed, zero config')


## Ex 4: Qdrant with Metadata Filtering


In [ ]:
# pip install qdrant-client
from qdrant_client import QdrantClient, models
import numpy as np

client = QdrantClient(':memory:')

client.create_collection(
    collection_name='docs',
    vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE)
)

# Upsert with metadata
client.upsert('docs', points=[
    models.PointStruct(id=1, vector=np.random.rand(768).tolist(),
        payload={'source': 'policy_2025.pdf', 'year': 2025}),
    models.PointStruct(id=2, vector=np.random.rand(768).tolist(),
        payload={'source': 'policy_2023.pdf', 'year': 2023}),
])

# Search WITH filter (filterable HNSW — during traversal)
results = client.query_points('docs',
    query=np.random.rand(768).tolist(), limit=1,
    query_filter=models.Filter(must=[
        models.FieldCondition(key='year', match=models.MatchValue(value=2025))
    ])
)
print(f'Filtered result: {results.points[0].payload}')
print('\nQdrant: filters DURING HNSW traversal, <10% latency impact')


## Ex 5: Distance Metrics Equivalence


In [ ]:
import numpy as np

def normalize(v):
    return v / np.linalg.norm(v)

a = normalize(np.random.rand(768))
b = normalize(np.random.rand(768))

cosine = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
dot = np.dot(a, b)
l2 = np.sqrt(np.sum((a - b)**2))
l2_from_dot = np.sqrt(2 - 2 * dot)

print('For L2-normalized vectors:')
print(f'  cosine(a,b)     = {cosine:.6f}')
print(f'  dot(a,b)        = {dot:.6f}')
print(f'  L2(a,b)         = {l2:.6f}')
print(f'  sqrt(2-2*dot)   = {l2_from_dot:.6f}')
print(f'\n  cosine == dot? {abs(cosine-dot) < 1e-10}')
print(f'  L2 == sqrt(2-2*dot)? {abs(l2-l2_from_dot) < 1e-10}')
print('\nUse dot product for speed (no normalization step)')


## Ex 6: HNSW Parameters


In [ ]:
print('HNSW Key Parameters:')
print('='*50)
params = [
    ('M', '16', '16-64', 'Connections per node. Higher=better recall, more memory.'),
    ('efConstruction', '200', '100-500', 'Build quality. Higher=better graph, slower build.'),
    ('efSearch', '100', '50-500', 'Query recall. Tunable without rebuild!'),
]
for name, default, range_, impact in params:
    print(f'  {name:18s} default={default:4s} range={range_:8s}')
    print(f'    {impact}')

print('\nMemory formula: N × (M×2×4 + 8) + N × D × 4 bytes')
N, M, D = 1_000_000, 16, 768
graph_bytes = N * (M * 2 * 4 + 8)
vec_bytes = N * D * 4
print(f'\n1M vectors, 768D, M=16:')
print(f'  Graph: {graph_bytes/1e9:.2f} GB')
print(f'  Vectors: {vec_bytes/1e9:.2f} GB')
print(f'  Total: {(graph_bytes+vec_bytes)/1e9:.2f} GB')


## Ex 7: Vector DB Decision Framework


In [ ]:
print('Vector Database Decision Framework:')
print('='*50)
dbs = [
    ('ChromaDB', 'Prototype', '<10M', 'Free', 'Zero config'),
    ('Qdrant', 'Production RAG', '<50M', 'Free 1GB', 'Best filtering (Rust)'),
    ('Weaviate', 'Multi-tenant SaaS', '<50M', '~$25/mo', 'Best hybrid search (Go)'),
    ('Milvus/Zilliz', 'Billion-scale', 'Billions', 'Free 5GB', 'GPU indexes, DiskANN'),
    ('LanceDB', 'Serverless/edge', 'Billions', 'Free OSS', 'S3-backed, zero server'),
    ('Pinecone', 'Zero ops', '100M+', 'Usage', 'Managed, 7ms p99'),
    ('pgvector', 'PostgreSQL users', '<5M', 'Free', 'No separate DB needed'),
    ('Turbopuffer', 'Cost-sensitive', 'Billions', '~$10/mo', 'S3-native, 10M+ writes/sec'),
]
for name, best, scale, price, note in dbs:
    print(f'  {name:15s} | {best:20s} | {scale:10s} | {price:10s} | {note}')


## Ex 8: Hybrid Search Pattern


In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

corpus = [
    'Error code ERR_SSL_PROTOCOL_ERROR on Chrome 120',
    'SSL certificate validation failed during HTTPS handshake',
    'Machine learning models require GPU for training',
]

# BM25 sparse retrieval
tokenized = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized)
query = 'ERR_SSL_PROTOCOL_ERROR'
bm25_scores = bm25.get_scores(query.lower().split())

# RRF fusion
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0) + 1/(k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])

sparse_ranking = list(np.argsort(-bm25_scores))
print(f'BM25 top: {corpus[sparse_ranking[0]][:50]}')
print(f'BM25 catches exact error codes that dense search misses!')
print(f'Hybrid: +26-31% NDCG over dense-only')


## Ex 9: India Embedding Considerations


In [ ]:
print('Indian Language Embedding Guide:')
print('='*50)
models = [
    ('bge-multilingual-gemma2', '+35% recall', '~3GB', 'Sarvam choice'),
    ('BGE-M3', 'Baseline', '~2GB', 'MIT, 100+ langs'),
    ('Gemini Embedding 2', 'Highest x-lingual', 'API', '$0.20/M'),
    ('bhasha-embed-v0', 'Romanized Hindi', 'Small', 'Code-mixed'),
    ('L3Cube-IndicSBERT', '10 Indian langs', '~1GB', 'MuRIL-based'),
]
print('\nModel Comparison:')
for name, perf, size, note in models:
    print(f'  {name:25s} | {perf:18s} | {size:6s} | {note}')

print('\nCross-lingual gap: 20-40% accuracy loss')
print('Mitigation: translate query to English + dual retrieval + RRF')
print('Tokenizer: standard BPE over-segments Devanagari 3-5×')
print('Self-host on AWS Mumbai r6g: ~$150/month for 10M vectors')


---
## Recap
Embedding model: Gemini #1 retrieval (68.32), Qwen3-8B #1 open-source (MMTEB 70.58, Apache 2.0). Matryoshka: 768 dims = 75% storage savings with minimal quality loss. Vector DB: Qdrant (<50M, best filtering), Weaviate (multi-tenant SaaS), Milvus (billions, GPU). HNSW universal default (M=16, efSearch=100); IVF-PQ for billions; DiskANN for SSD at $1/M vectors. Distance: cosine=dot=L2 on normalized vectors. Production: hybrid search (+26-31% NDCG), semantic caching (80-95% savings), embedding versioning. India: bge-multilingual-gemma2 (+35% over BGE-M3), self-host AWS Mumbai (~$150/mo), DPDP: embeddings = derived personal data.
